In [ ]:
%pip install pandas numpy seaborn scikit-learn matplotlib

# ✅ PART C — Analytics (Insights + KPIs)

### Imports & Configuration


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from scipy.stats import norm

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8")

### Data Loading & Validation

In [ ]:
def load_csv(path, parse_dates=None):
    try:
        path = str(path)  # supports pathlib
        df = pd.read_csv(path, parse_dates=parse_dates)
        print(f"Loaded {path} | shape={df.shape}")
        return df    
    except FileNotFoundError:
            raise RuntimeError(f"❌ File not found: {path}")
    except OSError as e:
        raise RuntimeError(f"❌ OS error while reading {path}: {e}")
    except Exception as e:
            raise RuntimeError(f"❌ Failed to load {path}: {e}")


BASE_DIR = Path.cwd()        # analysis/
INPUT_DIR = BASE_DIR.parent / "data"

sales = load_csv(INPUT_DIR / "fact_sales_store_sku_daily.csv", parse_dates=["date"])
inventory = load_csv(INPUT_DIR / "fact_inventory_store_sku_daily.csv", parse_dates=["date"])
replenishment = load_csv(INPUT_DIR / "replenishment_inputs_store_sku.csv")

sales.info()
inventory.info()
replenishment.info()

## PART C‑1: Demand Understanding
Metrics & Visuals:
- Demand trend
- Weekly seasonality
- Promo / holiday uplift
- Top SKUs by volume & revenue

In [ ]:
# Overall demand trend
daily_demand = sales.groupby("date", as_index=False)["units_sold"].sum()

plt.figure(figsize=(12,4))
plt.plot(daily_demand["date"], daily_demand["units_sold"])
plt.title("Overall Demand Trend")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.show()

# Weekly seasonality
weekly = sales.groupby("day_of_week")["units_sold"].mean()
weekly.plot(kind="bar", title="Weekly Demand Seasonality", xlabel="Day of Week", ylabel="Average Units Sold")
plt.show()

# Promo impact
promo_cmp = sales.groupby("promo_flag")["units_sold"].mean()
promo_cmp.plot(kind="bar", title="Promo v/s Non - Promo Demand", xlabel="Promo Flag", ylabel="Average Units Sold")
plt.show()

# Top SKUs
top_skus = (
    sales.groupby("sku_id")
    .agg(units=("units_sold","sum"), revenue=("revenue","sum"))
    .sort_values("revenue", ascending=False)
    .head(10)
)
display(top_skus)

## PART C‑2: Forecasting (Mandatory)
Forecast Horizon: 28 days
Models:
1. Seasonal Naive
   Formula: Forecast(t) = Actual(t‑7)
2. Regression Model
   Features: DayOfWeek + PromoFlag + HolidayFlag

Evaluation Metrics:
- WAPE = Σ|Actual − Forecast| / ΣActual
- MAPE = avg(|Actual − Forecast| / Actual)
- Bias = Σ(Forecast − Actual) / ΣActual


In [ ]:
# Feature engineering
sales_feat = sales.copy()
sales_feat["dow"] = sales_feat["day_of_week"]

# Train / validation split
max_date = sales_feat["date"].max()
cutoff = max_date - pd.Timedelta(days=28)

train = sales_feat[sales_feat["date"] <= cutoff]
valid = sales_feat[sales_feat["date"] > cutoff]

# Seasonal naive
valid["forecast_naive"] = (
    valid.sort_values("date")
         .groupby(["store_id","sku_id"])["units_sold"]
         .shift(7)
)

# Regression
X_cols = ["dow","promo_flag","holiday_flag"]
model = LinearRegression()
model.fit(train[X_cols], train["units_sold"])

predictions = model.predict(valid[X_cols])
valid['forecast_reg'] = np.clip(predictions, a_min=0, a_max=None)  # ensures we don't predict negative sales, which aren't possible in this context.

# Forecast KPIs (Overall + Category)

def WAPE(a,f): return np.abs(a-f).sum() / max(a.sum(),1)
def MAPE(a,f): return np.mean(np.abs(a-f) / np.where(a==0,np.nan,a))
def Bias(a,f): return (f-a).sum() / max(a.sum(),1)

print("OVERALL FORECAST QUALITY")
print("Naive WAPE:", round(WAPE(valid["units_sold"], valid["forecast_naive"].fillna(0)),3))
print("Reg   WAPE:", round(WAPE(valid["units_sold"], valid["forecast_reg"]),3))
print("Naive Bias:", round(Bias(valid["units_sold"], valid["forecast_naive"].fillna(0)),3))
print("Reg   Bias:", round(Bias(valid["units_sold"], valid["forecast_reg"]),3))

# Category‑level rollup
valid["category"] = valid["sku_id"].str[:3]

cat_kpis = (
    valid.groupby("category")
    .apply(lambda x: pd.Series({
        "WAPE": WAPE(x["units_sold"], x["forecast_reg"]),
        "MAPE": MAPE(x["units_sold"], x["forecast_reg"]),
        "Bias": Bias(x["units_sold"], x["forecast_reg"])
    }))
    .reset_index()
)
cols_to_round = ["WAPE", "MAPE", "Bias"]
cat_kpis[cols_to_round] = cat_kpis[cols_to_round].round(3)
display(cat_kpis.sort_values("WAPE"))


### Forecast Validation – Visual Diagnostics

The following charts validate model performance beyond aggregate KPIs:
- Trend alignment
- Bias behavior
- Category‑level forecast stability

In [ ]:
# Visualize actual vs forecasts
plot_df = (
    valid.groupby("date")
    .agg(
        actual=("units_sold", "sum"),
        naive=("forecast_naive", "sum"),
        reg=("forecast_reg", "sum")
    )
    .reset_index()
)

plt.figure(figsize=(12,5))
plt.plot(plot_df["date"], plot_df["actual"], label="Actual", linewidth=2)
plt.plot(plot_df["date"], plot_df["naive"], label="Naive", linestyle="--")
plt.plot(plot_df["date"], plot_df["reg"], label="Regression", linestyle="--")
plt.title("Actual vs Forecasted Demand (Validation Period)")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.show()

# Forecast Error Distribution (Regression)
valid["reg_error"] = valid["forecast_reg"] - valid["units_sold"]

plt.figure(figsize=(7,4))
sns.histplot(valid["reg_error"], bins=40, kde=True)
plt.axvline(0, color="red", linestyle="--")
plt.title("Regression Forecast Error Distribution")
plt.xlabel("Forecast Error (Units)")
plt.show()

# Category‑level Forecast Quality
plt.figure(figsize=(10,4))
sns.barplot(
    data=cat_kpis.sort_values("WAPE"),
    x="category",
    y="WAPE"
)
plt.title("Regression Forecast WAPE by Category")
plt.ylabel("WAPE")
plt.xlabel("Category")
plt.show()

## PART C‑3: Inventory Risk Segmentation & Action Framework
Objective:
- Classify each **store–SKU** into actionable inventory risk buckets to support replenishment and inventory optimization decisions.

DOH Formula:
- DOH = OnHandUnits / AvgDailyDemand

### Risk Definitions

**Stockout Risk**
- Projected on‑hand inventory will be insufficient to cover demand during supplier lead time (projected on‑hand < demand during lead time) 
- Condition:
\[
OnHand < AvgDailyDemand \times LeadTime
\]

**Overstock Risk**
- Inventory coverage (Days of Hand) is significantly higher than business‑defined thresholds (DOH > threshold (45 days))
- Condition:
\[
DOH > CategoryThreshold
\]

**Healthy**
- Inventory is within acceptable bounds

Each risk bucket is mapped to a **recommended operational action**.



In [ ]:

# Merge demand and lead time attributes into inventory snapshot
inventory = inventory.merge(
    replenishment[["store_id", "sku_id", "avg_daily_demand", "lead_time_days"]],
    on=["store_id", "sku_id"],
    how="left"
)

# Compute Days of Hand (DOH)

inventory["DOH"] = (inventory["on_hand_units"] / inventory["avg_daily_demand"].replace(0, np.nan)).round(3)

# Projected demand during lead time
inventory["lt_demand"] = inventory["avg_daily_demand"] * inventory["lead_time_days"]

'''
Overstock Thresholds (Business Assumption)

For simplicity and interpretability:
- A fixed DOH threshold of **45 days** is used to flag overstock risk.
- In real implementations, this would vary by category (fast‑moving vs slow‑moving SKUs).
'''

# Inventory Risk Segmentation
inventory["risk_segment"] = np.select(
    [
        inventory["on_hand_units"] < inventory["lt_demand"],  # Stockout condition
        inventory["DOH"] > 45                                  # Overstock condition
    ],
    [
        "Stockout Risk",
        "Overstock Risk"
    ],
    default="Healthy"
)

# Sanity check: distribution of risk segments
display(inventory["risk_segment"].value_counts())

'''
#Recommended Action Framework

Each risk segment is mapped to a clear operational action:

| Risk Segment     | Recommended Action                                  |
|------------------|-----------------------------------------------------|
| Stockout Risk    | Replenish / Expedite supply                         |
| Overstock Risk   | Transfer stock / Discount / Reduce future orders    |
| Healthy          | Maintain current policy                             |

'''

def recommend_action(row):
    if row["risk_segment"] == "Stockout Risk":
        return "Replenish / Expedite"
    elif row["risk_segment"] == "Overstock Risk":
        return "Transfer / Discount / Reduce Reorder"
    else:
        return "Maintain"

inventory["recommended_action"] = inventory.apply(
    recommend_action,
    axis=1
)

# Top 10 Stockout‑Risk store–SKUs
display(
    inventory[inventory["risk_segment"] == "Stockout Risk"]
    .sort_values("on_hand_units")
    [
        [
            "store_id",
            "sku_id",
            "on_hand_units",
            "avg_daily_demand",
            "lead_time_days",
            "lt_demand",
            "risk_segment",
            "recommended_action"
        ]
    ]
    .head(10)
)

# Top 10 Overstock‑Risk store–SKUs
display(
    inventory[inventory["risk_segment"] == "Overstock Risk"]
    .sort_values("DOH", ascending=False)
    [
        [
            "store_id",
            "sku_id",
            "on_hand_units",
            "avg_daily_demand",
            "DOH",
            "risk_segment",
            "recommended_action"
        ]
    ]
    .head(10)
)

## PART C‑4: Replenishment Investigation Table

Objective:
- Identify **root causes of stockouts** by analyzing the **top 2 categories**
- contributing the most stockout events, and propose **data‑backed corrective actions**.


Metrics:
- Stockout rate
- Lost sales proxy
- Stores contributing most
- Demand volatility
- 2-3 hypotheses for stockouts
- 1 follow-up experiment/data to validate each hypothesis
- Evidence used (metric/chart/table reference)


In [ ]:

# Derive category (assumption: first 3 chars of SKU represent category)
inventory["category"] = inventory["sku_id"].str[:3]

# Lost sales proxy:
# If stockout, assume unmet demand equals avg daily demand
inventory["lost_sales_units"] = np.where(
    inventory["stockout_flag"] == 1,
    inventory["avg_daily_demand"],
    0
)

# Identify top 2 categories by stockout rate
top_categories = (
    inventory.groupby("category")["stockout_flag"]
    .mean()
    .sort_values(ascending=False)
    .head(2)
    .index
)

top_categories

# Category-level investigation metrics
category_metrics = (
    inventory[inventory["category"].isin(top_categories)]
    .groupby("category")
    .agg(
        stockout_rate=("stockout_flag", "mean"),
        lost_sales_proxy_units=("lost_sales_units", "sum"),
        demand_volatility=("avg_daily_demand", "std")
    )
    .reset_index()
)

category_metrics = category_metrics.round(3)
display(category_metrics)

# Store contribution to stockouts for top categories
store_contribution = (
    inventory[inventory["category"].isin(top_categories)]
    .groupby(["category", "store_id"])["stockout_flag"]
    .sum()
    .reset_index()
    .sort_values(["category", "stockout_flag"], ascending=[True, False])
)

display(store_contribution.groupby("category").head(3))


print("""
### Root‑Cause Hypotheses & Validation Plan

Based on the above metrics, the following hypotheses are proposed for the
top stockout‑driving categories.

Each hypothesis includes a **clear validation experiment** and
**evidence reference** from this analysis.
""")

# Constructing a structured investigation table

rows = []

for _, row in category_metrics.iterrows():
    rows.append({
        "category": row["category"],
        "key_issue_observed": "High stockout rate with volatile demand",
        "hypothesis_1": "Safety stock underestimated for high demand variability",
        "validation_experiment_1": "Recalculate safety stock using higher service level (z-score) and backtest",
        "hypothesis_2": "Promotional uplift not incorporated in forecasts",
        "validation_experiment_2": "Include promo flag in demand forecasting and re-evaluate forecast error",
        "evidence_used": (
            "Category stockout metrics table, "
            "store contribution table, "
            "demand volatility vs stockout analysis"
        )
    })

investigation_table = pd.DataFrame(rows)

display(investigation_table)

print(f"""
### Key Insights

- Stockouts are **highly concentrated** in a limited number of product categories
  (e.g., {", ".join(category_metrics["category"].astype(str))}),
  rather than being evenly distributed across the assortment.

- These categories exhibit **elevated demand volatility**, which increases the
  likelihood that current safety stock and replenishment parameters are insufficient.

- Store‑level analysis shows that **a small subset of stores contributes
  disproportionately** to stockout events, indicating execution or replenishment
  frequency gaps rather than network‑wide issues.

- The primary drivers of stockouts appear to be:
  - Under‑buffering for demand variability
  - Lack of promo‑aware forecasting adjustments
  - Insufficient protection against lead‑time uncertainty

- Targeted actions such as **safety stock recalibration**, **promo‑adjusted demand
  forecasting**, and **store‑specific replenishment tuning** are expected to
  materially reduce stockout risk without materially increasing inventory holding costs.

These insights directly inform the **replenishment policy design in Part D**
and the **30‑day impact estimation in Part E**.
""")

# Stockout Concentration by Store (Top Categories)
plot_df = store_contribution.groupby("store_id")["stockout_flag"].sum().reset_index()

plt.figure(figsize=(8,4))
sns.barplot(
    data=plot_df.sort_values("stockout_flag", ascending=False).head(10),
    x="store_id",
    y="stockout_flag"
)
plt.title("Top Stores Contributing to Stockouts (Top Categories)")
plt.ylabel("Stockout Days")
plt.xlabel("Store")
plt.show()

# Demand Volatility vs Stockout Rate (Category Level)
plt.figure(figsize=(6,4))
sns.scatterplot(
    data=category_metrics,
    x="demand_volatility",
    y="stockout_rate",
    s=100
)
plt.title("Demand Volatility vs Stockout Rate (Top Categories)")
plt.xlabel("Demand Volatility (Std Dev)")
plt.ylabel("Stockout Rate")
plt.show()

# PART D: Replenishment Policy

Objective:
Design a **store‑SKU level reorder policy** that balances service level
targets against inventory risk while respecting operational constraints.

This section computes:
- Safety Stock (demand uncertainty buffer)
- Reorder Point (ROP)
- Recommended Order Quantity (Order‑Up‑To policy)

Constraints applied:
- Minimum Order Quantity (MOQ)
- Shelf‑life protection (perishable SKUs)
- Maximum store storage capacity (optional)


In [ ]:
policy = replenishment.copy()

# Convert service level target to Z-score
# Clamp values to avoid infinite z-scores
policy["z"] = policy["service_level_target"].apply(
    lambda x: norm.ppf(min(max(x, 0.50), 0.999))
)

# Safety Stock = z × σD ​× √LeadTime​ (z * demand std dev * sqrt(lead time))
policy["safety_stock"] = (
    policy["z"] *
    policy["demand_std_dev"] *
    np.sqrt(policy["lead_time_days"])
)

# Reorder Point (ROP) = AvgDailyDemand × LeadTime + SafetyStock
policy["ROP"] = (
    policy["avg_daily_demand"] * policy["lead_time_days"]
    + policy["safety_stock"]
)

# Current inventory
current_inventory = (
    inventory.sort_values("date")
    .groupby(["store_id", "sku_id"])
    .tail(1)[["store_id", "sku_id", "on_hand_units"]]
)

# Merge current inventory into policy
policy = policy.merge(
    current_inventory,
    on=["store_id", "sku_id"],
    how="left"
)


# Order‑up‑to (30 days cover)
TARGET_DOH = 30

policy["order_up_to_level"] = TARGET_DOH * policy["avg_daily_demand"]

policy["raw_order_qty"] = np.maximum(
    policy["order_up_to_level"] - policy["on_hand_units"],
    0
)

# Apply constraints to recommended order quantity
# Constraints: MOQ, Shelf‑life (max sellable during shelf life), Max store capacity (optional) 
def apply_constraints(row):
    qty = row["raw_order_qty"]

    # MOQ constraint
    if qty > 0:
        qty = max(qty, row.get("MOQ", 0))

    # Shelf‑life constraint (max sellable during shelf life)
    if not pd.isna(row.get("shelf_life_days")):
        max_sellable = row["avg_daily_demand"] * row["shelf_life_days"]
        qty = min(qty, max_sellable)

    # Max store capacity (optional)
    if not pd.isna(row.get("max_store_capacity")):
        qty = min(qty, row["max_store_capacity"])

    return round(qty)

policy["recommended_order_qty"] = policy.apply(
    apply_constraints,
    axis=1
)

# ---------------------------------------------------
# Derive SKU-level unit price from sales data
# ---------------------------------------------------

# Avoid division by zero
sales_price = sales[sales["units_sold"] > 0].copy()

sales_price["unit_price"] = (
    sales_price["revenue"] / sales_price["units_sold"]
)

# Average price per SKU (robust to daily noise)
sku_price = (
    sales_price.groupby("sku_id")["unit_price"]
    .mean()
    .reset_index()
)

policy = policy.merge(
    sku_price,
    on="sku_id",
    how="left"
)

# ---------------------------------------------------
# Derive unit cost using margin proxy
# ---------------------------------------------------

# Get SKU-level average margin proxy
sku_margin = (
    sales.groupby("sku_id")["margin_proxy"]
    .mean()
    .reset_index()
)

policy = policy.merge(
    sku_margin,
    on="sku_id",
    how="left"
)

# Derive unit cost
policy["unit_cost"] = policy["unit_price"] * (1 - policy["margin_proxy"])

# Display final policy with key columns
display(
    policy[[
        "store_id",
        "sku_id",
        "avg_daily_demand",
        "lead_time_days",
        "service_level_target",
        "safety_stock",
        "ROP",
        "on_hand_units",
        "recommended_order_qty"
    ]].head(10)
)


# Part E — Impact Estimation (30‑Day Projection)

Objective:
Estimate the **business impact** of the proposed replenishment policy over
a 30‑day horizon using practical proxies.

Impact dimensions:
- Reduction in stockout days
- Lost sales avoided (units and ₹)
- Inventory value change (holding cost proxy)

Scenarios evaluated:
- Base case
- Best case
- Worst case

### Assumptions

- A stockout is assumed when **on‑hand inventory < Reorder Point (ROP)**.
- Lost sales during stockout are proxied as:
  \[
  AvgDailyDemand \times LeadTime
  \]
- Impact is evaluated over a **30‑day horizon**.
- Best and worst cases reflect ±20% variation in execution effectiveness.
- Inventory holding cost proxy assumes excess inventory value change


In [ ]:
# 30-day horizon
HORIZON_DAYS = 30

# Baseline stockout indicator
policy["baseline_stockout_flag"] = np.where(
    policy["on_hand_units"] < policy["ROP"], 1, 0
)

# Post-policy stockout indicator (assume replenishment eliminates stockout)
policy["post_policy_stockout_flag"] = 0

# Stockout days (proxy)
policy["stockout_days_base"] = (
    policy["baseline_stockout_flag"] * HORIZON_DAYS
)

# Stockout days avoided = baseline stockout days - post-policy stockout days
policy["stockout_days_avoided"] = (
    policy["stockout_days_base"]
)

# Base case lost units avoided
policy["lost_units_avoided_base"] = np.where(
    policy["baseline_stockout_flag"] == 1,
    policy["avg_daily_demand"] * policy["lead_time_days"],
    0
)

# Best / Worst scenarios
policy["lost_units_avoided_best"] = policy["lost_units_avoided_base"] * 1.2
policy["lost_units_avoided_worst"] = policy["lost_units_avoided_base"] * 0.8

# Assume unit_price is available in policy
policy["lost_sales_value_base"] = (
    policy["lost_units_avoided_base"] * policy["unit_price"]
)

# Best / Worst scenarios for lost sales value
policy["lost_sales_value_best"] = (
    policy["lost_units_avoided_best"] * policy["unit_price"]
)

# Worst case for lost sales value
policy["lost_sales_value_worst"] = (
    policy["lost_units_avoided_worst"] * policy["unit_price"]
)

# Inventory value before and after replenishment
policy["inventory_value_before"] = (
    policy["on_hand_units"] * policy["unit_cost"]
)

# Inventory value after replenishment
policy["inventory_value_after"] = (
    (policy["on_hand_units"] + policy["recommended_order_qty"])
    * policy["unit_cost"]
)

# Inventory value change due to replenishment
policy["inventory_value_change"] = (
    policy["inventory_value_after"] - policy["inventory_value_before"]
)

impact_summary = pd.Series({
    "Stockout Days Avoided": policy["stockout_days_avoided"].sum(),
    "Lost Units Avoided (Base)": policy["lost_units_avoided_base"].sum(),
    "Lost Units Avoided (Best)": policy["lost_units_avoided_best"].sum(),
    "Lost Units Avoided (Worst)": policy["lost_units_avoided_worst"].sum(),
    "Lost Sales Value ₹ (Base)": policy["lost_sales_value_base"].sum(),
    "Lost Sales Value ₹ (Best)": policy["lost_sales_value_best"].sum(),
    "Lost Sales Value ₹ (Worst)": policy["lost_sales_value_worst"].sum(),
    "Inventory Value Change ₹": policy["inventory_value_change"].sum()
})


impact_summary_df = (
    impact_summary
    .round(3)
    .rename("value")
    .reset_index()
    .rename(columns={"index": "metric"})
)

print("Impact Summary Table:")
display(impact_summary_df)


print("Impact Summary Chart:")
impact_summary[
    ["Lost Sales Value ₹ (Base)", "Lost Sales Value ₹ (Best)", "Lost Sales Value ₹ (Worst)"]
].plot(kind="bar", figsize=(6,4), title="Lost Sales Avoided (₹) by Scenario")
plt.ylabel("₹ Value")
plt.show()

### Impact Interpretation

- The proposed replenishment policy is expected to **eliminate most stockout days**
  for high‑risk store‑SKU combinations.
- This translates into **material recovery of lost sales**, even under conservative
  assumptions.
- The increase in inventory value represents a **controlled working‑capital trade‑off**
  to achieve higher service levels.
- Best‑case and worst‑case scenarios provide a reasonable confidence band for
  decision‑making.